In [1]:
!pip install kaleido==0.2.1
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.0 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [3]:
import os
import glob
import zipfile
import math
import copy
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from tqdm import tqdm

# اتصال به گوگل درایو (مخصوص محیط کولب)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("Not running in Google Colab. Make sure your paths are set correctly.")

# ==========================================
# تنظیمات اولیه و مسیرها
# ==========================================
DRIVE_ZIP_PATH = "/content/drive/MyDrive/kitti_sample.zip"
BASE_DIR = "/content/kitti_data"
EXTRACT_DIR = os.path.join(BASE_DIR, "extracted")
OUTPUT_FIG_DIR = "/content/output_figures"

os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)

# ==========================================
# بخش 0: استخراج هوشمند دیتاست Odometry
# ==========================================
def extract_kitti_from_drive():
    if not os.path.exists(DRIVE_ZIP_PATH):
        raise FileNotFoundError(f"Error: The file {DRIVE_ZIP_PATH} was not found in your Google Drive.")

    if not os.path.exists(EXTRACT_DIR):
        print(f"Extracting files from Google Drive: {DRIVE_ZIP_PATH} ...")
        with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_DIR)
        print("Extraction completed!")
    else:
        print("Files are already extracted in Colab's local storage.")

    # جستجوی هوشمند بر اساس محتوای داخل زیپ شما
    velodyne_path = None
    pose_file = None

    for root, dirs, files in os.walk(EXTRACT_DIR):
        # اگر در این پوشه فایل .bin وجود داشت، پس اینجا پوشه لایدار است
        if any(f.endswith('.bin') for f in files):
            velodyne_path = root

        # پیدا کردن فایل pose بر اساس نام دقیق آن در عکس شما
        if 'gt_pose_00.txt' in files:
            pose_file = os.path.join(root, 'gt_pose_00.txt')

    if velodyne_path is None or pose_file is None:
        print("Warning: Could not find the .bin files or 'gt_pose_00.txt'.")

    print(f"Found Velodyne data at: {velodyne_path}")
    print(f"Found Pose data at: {pose_file}")

    return velodyne_path, pose_file

# ==========================================
# توابع کمکی
# ==========================================
def load_kitti_bin(bin_path, voxel_size=None):
    scan = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 4)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(scan[:, 0:3])
    if voxel_size:
        pcd = pcd.voxel_down_sample(voxel_size=voxel_size)
    return pcd

# استخراج مسیر از ماتریس 3x4
def load_groundtruth_odometry(pose_file, num_frames):
    gt_trajectory = []
    with open(pose_file, 'r') as f:
        lines = f.readlines()
        for i in range(min(num_frames, len(lines))):
            vals = np.array([float(x) for x in lines[i].split()])
            # استخراج بردار انتقال (x, y, z) از ستون آخر ماتریس 3x4
            tx, ty, tz = vals[3], vals[7], vals[11]
            gt_trajectory.append([tx, ty, tz])

    gt_trajectory = np.array(gt_trajectory)
    return gt_trajectory - gt_trajectory[0]

# ==========================================
# بخش 1: ثبت دو ابر نقطه
# ==========================================
def task1_point_cloud_registration(bin_files):
    print("\n--- Task 1: Point Cloud Registration ---")
    source_pcd = load_kitti_bin(bin_files[0], voxel_size=0.5)
    target_pcd = load_kitti_bin(bin_files[5], voxel_size=0.5)

    def visualize_and_save(src, tgt, trans_mat, title, filename):
        src_temp = copy.deepcopy(src).transform(trans_mat)
        s_pts = np.asarray(src_temp.points)
        t_pts = np.asarray(tgt.points)

        step = 2

        fig = go.Figure(data=[
            go.Scatter3d(x=t_pts[::step, 0], y=t_pts[::step, 1], z=t_pts[::step, 2],
                         mode='markers', marker=dict(size=1.5, color='blue'), name='Target'),
            go.Scatter3d(x=s_pts[::step, 0], y=s_pts[::step, 1], z=s_pts[::step, 2],
                         mode='markers', marker=dict(size=1.5, color='red'), name='Source (Transformed)')
        ])
        fig.update_layout(title=title, scene=dict(aspectmode='data'))
        fig.write_image(os.path.join(OUTPUT_FIG_DIR, filename), width=1000, height=700, scale=2)
        print(f"Saved: {filename}")

    visualize_and_save(source_pcd, target_pcd, np.identity(4), "Before ICP (Unregistered)", "task1_before_icp.png")

    reg_p2p = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd, max_correspondence_distance=2.0, init=np.identity(4),
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100)
    )

    visualize_and_save(source_pcd, target_pcd, reg_p2p.transformation, "After ICP (Aligned)", "task1_after_icp.png")

# ==========================================
# بخش 2: ثبت افزایشی
# ==========================================
def task2_incremental_registration(bin_files, num_frames=30):
    print(f"\n--- Task 2: Incremental Map Building ({num_frames} frames) ---")
    global_map = load_kitti_bin(bin_files[0], voxel_size=0.5)
    current_transform = np.identity(4)
    trajectory = [current_transform[:3, 3]]

    for i in tqdm(range(1, num_frames), desc="Fusing Point Clouds"):
        source_pcd = load_kitti_bin(bin_files[i], voxel_size=0.5)
        target_pcd = load_kitti_bin(bin_files[i-1], voxel_size=0.5)

        reg_p2p = o3d.pipelines.registration.registration_icp(
            source_pcd, target_pcd, max_correspondence_distance=2.0, init=np.identity(4),
            estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(),
            criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=50)
        )

        current_transform = np.dot(current_transform, reg_p2p.transformation)
        trajectory.append(current_transform[:3, 3])

        src_global = copy.deepcopy(source_pcd).transform(current_transform)
        global_map += src_global
        global_map = global_map.voxel_down_sample(voxel_size=0.5)

    trajectory = np.array(trajectory)

    pts = np.asarray(global_map.points)
    step = max(1, len(pts) // 50000)
    fig = go.Figure(data=[
        go.Scatter3d(x=pts[::step, 0], y=pts[::step, 1], z=pts[::step, 2], mode='markers', marker=dict(size=1.0, color='gray', opacity=0.5), name='3D Map'),
        go.Scatter3d(x=trajectory[:, 0], y=trajectory[:, 1], z=trajectory[:, 2], mode='lines+markers', line=dict(color='red', width=4), name='Path')
    ])
    fig.update_layout(title="Unified 3D Map & Trajectory", scene=dict(aspectmode='data'))
    fig.write_image(os.path.join(OUTPUT_FIG_DIR, "task2_unified_map.png"), width=1200, height=800, scale=2)
    print("Saved: task2_unified_map.png")

    return trajectory

# ==========================================
# بخش 3: ارزیابی مسیر
# ==========================================
def task3_path_evaluation(estimated_trajectory, pose_file, num_frames=30):
    print("\n--- Task 3: Path Evaluation (FDE & ADE) ---")
    gt_trajectory = load_groundtruth_odometry(pose_file, num_frames)

    est_2d = estimated_trajectory[:num_frames, :2]
    gt_2d = gt_trajectory[:, [0, 2]]

    H = np.dot(gt_2d.T, est_2d)
    U, S, Vt = np.linalg.svd(H)
    R = np.dot(Vt.T, U.T)
    if np.linalg.det(R) < 0:
        Vt[1, :] *= -1
        R = np.dot(Vt.T, U.T)

    aligned_gt_2d = np.dot(gt_2d, R.T)

    distances = np.sqrt(np.sum((est_2d - aligned_gt_2d)**2, axis=1))
    ade = np.mean(distances)
    fde = distances[-1]

    print(f"-> ADE (Average Displacement Error): {ade:.4f} meters")
    print(f"-> FDE (Final Displacement Error):   {fde:.4f} meters")

    fig = go.Figure(data=[
        go.Scatter(x=aligned_gt_2d[:, 0], y=aligned_gt_2d[:, 1], mode='lines+markers', name='Aligned Groundtruth', line=dict(color='green', dash='dash')),
        go.Scatter(x=est_2d[:, 0], y=est_2d[:, 1], mode='lines+markers', name='Estimated Path (ICP)', line=dict(color='red'))
    ])
    fig.update_layout(title=f"Trajectory Alignment (ADE: {ade:.2f}m, FDE: {fde:.2f}m)", yaxis=dict(scaleanchor="x", scaleratio=1))
    fig.write_image(os.path.join(OUTPUT_FIG_DIR, "task3_trajectory_evaluation.png"), width=1000, height=600, scale=2)
    print("Saved: task3_trajectory_evaluation.png")

# ==========================================
# اجرای اصلی برنامه
# ==========================================
if __name__ == "__main__":
    velodyne_dir, pose_file = extract_kitti_from_drive()

    if velodyne_dir and pose_file:
        bin_files = sorted(glob.glob(os.path.join(velodyne_dir, '*.bin')))

        if len(bin_files) > 0:
            frames_to_process = 30

            task1_point_cloud_registration(bin_files)
            est_traj = task2_incremental_registration(bin_files, num_frames=frames_to_process)
            task3_path_evaluation(est_traj, pose_file, num_frames=frames_to_process)

            print("\n" + "="*50)
            print(f"🎉 All tasks completed! Figures are saved in: {OUTPUT_FIG_DIR}")
            print("="*50)

            try:
                from google.colab import files
                import shutil

                print("\nZipping and downloading the figures directly to your computer...")
                zip_filename = '/content/project3_figures'
                shutil.make_archive(zip_filename, 'zip', OUTPUT_FIG_DIR)
                files.download(f"{zip_filename}.zip")
            except ImportError:
                print("Google Colab specific download skipped.")

Mounted at /content/drive
Extracting files from Google Drive: /content/drive/MyDrive/kitti_sample.zip ...
Extraction completed!
Found Velodyne data at: /content/kitti_data/extracted/kitti_sample/00
Found Pose data at: /content/kitti_data/extracted/kitti_sample/gt_pose_00.txt

--- Task 1: Point Cloud Registration ---
Saved: task1_before_icp.png
Saved: task1_after_icp.png

--- Task 2: Incremental Map Building (30 frames) ---


Fusing Point Clouds: 100%|██████████| 29/29 [00:14<00:00,  1.94it/s]


Saved: task2_unified_map.png

--- Task 3: Path Evaluation (FDE & ADE) ---
-> ADE (Average Displacement Error): 1.1442 meters
-> FDE (Final Displacement Error):   1.1436 meters
Saved: task3_trajectory_evaluation.png

🎉 All tasks completed! Figures are saved in: /content/output_figures

Zipping and downloading the figures directly to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>